### Delhi Climate Time Series

1. Setup

In [1]:
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import root_mean_squared_error

from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestRegressor
from sklearn.svm import SVR

2. Load Dataset

In [2]:
df = pd.read_csv('../../data/delhi-climate.csv')
df.head()

,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.913043,85.869565,2.743478,59.000000
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333


In [3]:
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values('date')
df.head()

,date,meantemp,humidity,wind_speed,meanpressure
0,2017-01-01,15.913043,85.869565,2.743478,59.000000
1,2017-01-02,18.500000,77.222222,2.894444,1018.277778
2,2017-01-03,17.111111,81.888889,4.016667,1018.333333
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333


3. Feature Engineering
- We create lag features

In [4]:
df['lag_1'] = df['meantemp'].shift(1)
df['lag_2'] = df['meantemp'].shift(2)
df['lag_3'] = df['meantemp'].shift(3)

df = df.dropna()

In [5]:
df.head()

,date,meantemp,humidity,wind_speed,meanpressure,lag_1,lag_2,lag_3
3,2017-01-04,18.700000,70.050000,4.545000,1015.700000,17.111111,18.500000,15.913043
4,2017-01-05,18.388889,74.944444,3.300000,1014.333333,18.700000,17.111111,18.500000
5,2017-01-06,19.318182,79.318182,8.681818,1011.772727,18.388889,18.700000,17.111111
6,2017-01-07,14.708333,95.833333,10.041667,1011.375000,19.318182,18.388889,18.700000
7,2017-01-08,15.684211,83.526316,1.950000,1015.550000,14.708333,19.318182,18.388889


4. Define X and y

In [6]:
features = ['lag_1', 'lag_2', 'lag_3', 'humidity', 'wind_speed', 'meanpressure']
target = 'meantemp'

X = df[features]
y = df[target]

5. Train-Test Split (No shuffling in Time-Series)

In [7]:
split = int(len(df) * 0.8)

X_train, X_test = X[split:], X[:split]
y_train, y_test = y[split:], y[:split]

6. Scaling

In [8]:
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test = scaler.fit_transform(X_test)

7. ML Models

In [9]:
# Linear Regression
lr = LinearRegression()
lr.fit(X_train, y_train)

pred_lr = lr.predict(X_test)

In [10]:
rf = RandomForestRegressor(n_estimators=200)
rf.fit(X_train, y_train)

pred_rf = rf.predict(X_test)

In [11]:
svr = SVR()
svr.fit(X_train, y_train)

pred_svr = svr.predict(X_test)

8. Evaluation Function

In [12]:
results = {}

def evaluate(y_test, y_pred, name):
    rmse = root_mean_squared_error(y_test, y_pred)
    results[name] = rmse
    print(f"{name} RMSE: {rmse:0.4f}")

In [13]:
evaluate(y_test, pred_lr, "Linear Regression")
evaluate(y_test, pred_rf, "Random Forest Reg")
evaluate(y_test, pred_svr, "Support Vector Regressor")

Linear Regression RMSE: 11.6573
Random Forest Reg RMSE: 11.6846
Support Vector Regressor RMSE: 12.0204


In [14]:
best_model_name = min(results, key=results.get)
print("Best Model:", best_model_name)

Best Model: Linear Regression


In [15]:
import joblib

if best_model_name == "Linear Regression":
    joblib.dump(lr, "../../models/time_series_model.pkl")

elif best_model_name == "Random Forest":
    joblib.dump(rf, "../../models/time_series_model.pkl")

elif best_model_name == "SVR":
    joblib.dump(svr, "../../models/time_series_model.pkl")